# DEO Search Evaluation

Prepare data in BigQuery vector store and compare standard similarity search (Baseline) against DEO negation-aware search.

## 1. Setup

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# Verify environment variables
print(f"BIGQUERY_PROJECT_ID: {os.getenv('BIGQUERY_PROJECT_ID')}")
print(f"BIGQUERY_LOCATION:   {os.getenv('BIGQUERY_LOCATION')}")
print(f"BIGQUERY_DATASET:    {os.getenv('BIGQUERY_DATASET')}")
print(f"BIGQUERY_TABLE:      {os.getenv('BIGQUERY_TABLE')}")
print(f"EMBEDDING_MODEL:     {os.getenv('EMBEDDING_MODEL_NAME', 'gemini-embedding-001')}")

## 2. Load documents from JSONL

In [ ]:
import sys

# Add data_ingestion to the path so we can import ingest module
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("")), "data_ingestion"))

from ingest import load_documents_from_jsonl

SOURCE_DIR = "../source_documents/beir/nsir"

docs = load_documents_from_jsonl(SOURCE_DIR)

Found 1 JSONL file(s) in ../source_documents/beir/nsir
  Reading ../source_documents/beir/nsir/corpus.jsonl...
Loaded 3946 documents.


In [ ]:
# Inspect sample documents
for doc in docs[:3]:
    print(f"ID: {doc.metadata['_id']}")
    print(f"Text: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print("---")

ID: 10000
Text: Aaron and his sons to the priesthood, and arrayed them in the robes of office (Leviticus 8; cf. Exodus 28-29). He also related to them God's detailed instructions for performing their duties while the...
Metadata: {'_id': '10000', 'source': '../source_documents/beir/nsir/corpus.jsonl'}
---
ID: 10001
Text: Aaron Aaron ( or ; ""Ahärôn"") is a prophet, high priest, and the brother of Moses in the Abrahamic religions. Knowledge of Aaron, along with his brother Moses, comes exclusively from religious texts,...
Metadata: {'_id': '10001', 'source': '../source_documents/beir/nsir/corpus.jsonl'}
---
ID: 10002
Text: as songwriters, as well as the band itself. In late 1973, they were invited by Swedish television to contribute a song for the Melodifestivalen 1974 and from a number of new songs, the upbeat song "Wa...
Metadata: {'_id': '10002', 'source': '../source_documents/beir/nsir/corpus.jsonl'}
---


In [ ]:
# Check document stats
lengths = [len(doc.page_content) for doc in docs]
print(f"Total documents: {len(docs)}")
print(f"Text length - min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths) / len(lengths):.0f}")

## 3. (Optional) Load documents from directory (md/txt)

In [ ]:
# from ingest import load_and_split_documents
#
# docs = load_and_split_documents("../source_documents")

## 4. Ingest a small batch to BigQuery VectorStore

In [ ]:
from ingest import add_to_vector_store

# Test with a small subset first
test_docs = docs[:5]
print(f"Testing with {len(test_docs)} documents...")

add_to_vector_store(
    docs=test_docs,
    project_id=os.getenv("BIGQUERY_PROJECT_ID"),
    location=os.getenv("BIGQUERY_LOCATION"),
    dataset=os.getenv("BIGQUERY_DATASET"),
    table_name=os.getenv("BIGQUERY_TABLE", "vector_store"),
    batch_size=5,
)

## 5. Verify: query the vector store

In [ ]:
from langchain_google_community import BigQueryVectorStore
from langchain_google_vertexai import VertexAIEmbeddings

embeddings = VertexAIEmbeddings(model_name=os.getenv("EMBEDDING_MODEL_NAME", "gemini-embedding-001"))

vector_store = BigQueryVectorStore(
    project_id=os.getenv("BIGQUERY_PROJECT_ID"),
    location=os.getenv("BIGQUERY_LOCATION"),
    dataset_name=os.getenv("BIGQUERY_DATASET"),
    table_name=os.getenv("BIGQUERY_TABLE", "vector_store"),
    embedding=embeddings,
)

results = vector_store.similarity_search("What is ABBA?", k=3)
for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(f"Text: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")

--- Result 1 ---
Text: Studio Recording of 1975". In the United States, "Fernando" reached the Top 10 of the Cashbox Top 100 singles chart and number 13 on the "Billboard" Hot 100. It also topped the "Billboard" Adult Conte...
Metadata: {'doc_id': 'f6d1dbe0a8f54a449a6357c203329b2f', '_id': '11269', 'source': 'source_documents/beir/nsir/corpus.jsonl', 'score': 0.7580912012345873}

--- Result 2 ---
Text: Queen" and "Take A Chance On Me" were certified gold by the Recording Industry Association of America for sales of over one million copies each. The group also had 12 Top 20 singles on the "Billboard"...
Metadata: {'doc_id': '6a2056ce29a84676bd8afbc4aa618cb9', '_id': '12683', 'source': 'source_documents/beir/nsir/corpus.jsonl', 'score': 0.7614361068021904}

--- Result 3 ---
Text: very successful, earning her four Gold Records in the UK (where it peaked at number-six), Australia, Germany and Sweden. In the UK and Australia, "A" was in the top 100 albums of 2013. The same year t...
Metadat

## 6. Ingest all documents

In [ ]:
# Uncomment to ingest all documents
# add_to_vector_store(
#     docs=docs,
#     project_id=os.getenv("BIGQUERY_PROJECT_ID"),
#     location=os.getenv("BIGQUERY_LOCATION"),
#     dataset=os.getenv("BIGQUERY_DATASET"),
#     table_name=os.getenv("BIGQUERY_TABLE", "vector_store"),
#     batch_size=50,
# )

## 7. Compare: Baseline vs DEO Search

Compare `search_documents_in_bigquery` (standard similarity search) with `deo_search_documents_in_bigquery` (negation-aware DEO search) using queries from the BEIR NSIR dataset.

In [ ]:
import sys
import logging
from textwrap import shorten

# Add the parent directory to the path so we can import deo_rag_with_bigquery package
sys.path.insert(0, os.path.dirname(os.path.abspath("")))

from langchain_google_vertexai import VertexAIEmbeddings
from langchain_google_community import BigQueryVectorStore
from deo_rag_with_bigquery.deo_optimizer import DEOOptimizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize shared resources
embedding_model = VertexAIEmbeddings(
    model_name=os.getenv("EMBEDDING_MODEL_NAME", "gemini-embedding-001")
)

vector_store = BigQueryVectorStore(
    project_id=os.getenv("BIGQUERY_PROJECT_ID"),
    location=os.getenv("BIGQUERY_LOCATION"),
    dataset_name=os.getenv("BIGQUERY_DATASET"),
    table_name=os.getenv("BIGQUERY_TABLE", "vector_store"),
    embedding=embedding_model,
)

optimizer = DEOOptimizer(embedding_model)

print("Shared resources initialized.")

In [ ]:
def baseline_search(query: str, k: int = 4):
    """Standard similarity search (no negation handling)."""
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )
    return retriever.invoke(query)


def deo_search(query: str, positives: list[str], negatives: list[str], k: int = 4):
    """DEO-optimized search (negation-aware)."""
    optimized_embedding = optimizer.optimize(
        query=query,
        positives=positives,
        negatives=negatives,
    )
    return vector_store.similarity_search_by_vector(optimized_embedding, k=k)


def compare_results(query: str, positives: list[str], negatives: list[str], k: int = 4):
    """Run both searches and display results side by side."""
    print("=" * 80)
    print(f"QUERY: {query}")
    print(f"  Positives: {positives}")
    print(f"  Negatives: {negatives}")
    print("=" * 80)

    baseline_docs = baseline_search(query, k=k)
    deo_docs = deo_search(query, positives, negatives, k=k)

    print(f"\n{'[Baseline Search]':^40} | {'[DEO Search]':^40}")
    print("-" * 80)

    max_len = max(len(baseline_docs), len(deo_docs))
    for i in range(max_len):
        left = ""
        if i < len(baseline_docs):
            doc_id = baseline_docs[i].metadata.get("_id", "?")
            text = shorten(baseline_docs[i].page_content, width=60, placeholder="...")
            left = f"[{doc_id}] {text}"

        right = ""
        if i < len(deo_docs):
            doc_id = deo_docs[i].metadata.get("_id", "?")
            text = shorten(deo_docs[i].page_content, width=60, placeholder="...")
            right = f"[{doc_id}] {text}"

        print(f"  {i+1}. {left}")
        print(f"     {'→ DEO:'} {right}")
        print()

    # Show overlap
    baseline_ids = {d.metadata.get("_id") for d in baseline_docs}
    deo_ids = {d.metadata.get("_id") for d in deo_docs}
    overlap = baseline_ids & deo_ids
    only_baseline = baseline_ids - deo_ids
    only_deo = deo_ids - baseline_ids
    print(f"  Overlap: {len(overlap)} docs | Baseline-only: {len(only_baseline)} | DEO-only: {len(only_deo)}")
    print()

    return baseline_docs, deo_docs

### Example 1: "Aaron's profile, but don't mention Moses."

A simple negation query from the NSIR dataset. The baseline search will likely return documents about both Aaron and Moses, while DEO should push Moses-related content away.

In [ ]:
baseline_1, deo_1 = compare_results(
    query="Aaron's profile, but don't mention Moses.",
    positives=[
        "Aaron biblical figure",
        "Aaron high priest",
        "Aaron in Abrahamic religions",
    ],
    negatives=[
        "Moses",
        "brother of Moses",
        "Moses and Aaron together",
    ],
    k=5,
)

QUERY: Aaron's profile, but don't mention Moses.
  Positives: ['Aaron biblical figure', 'Aaron high priest', 'Aaron in Abrahamic religions']
  Negatives: ['Moses', 'brother of Moses', 'Moses and Aaron together']


INFO:deo_rag_with_bigquery.deo_optimizer:  DEO step   0 | loss=-0.0505 pos=0.6908 reg=0.0000 neg=0.7413
INFO:deo_rag_with_bigquery.deo_optimizer:  DEO step  19 | loss=-0.1606 pos=0.5233 reg=0.5911 neg=0.8021



           [Baseline Search]             |               [DEO Search]              
--------------------------------------------------------------------------------
  1. [10001] Aaron Aaron ( or ; ""Ahärôn"") is a prophet, high priest,...
     → DEO: [10001] Aaron Aaron ( or ; ""Ahärôn"") is a prophet, high priest,...

  2. [11182] for here. A 14th-century Mamluk mosque stands here with...
     → DEO: [10000] Aaron and his sons to the priesthood, and arrayed them in...

  3. [10359] Moses, took his censer and stood between the living and...
     → DEO: [10313] is commemorated as one of the Holy Forefathers in the...

  4. [10000] Aaron and his sons to the priesthood, and arrayed them in...
     → DEO: [11182] for here. A 14th-century Mamluk mosque stands here with...

  5. [10313] is commemorated as one of the Holy Forefathers in the...
     → DEO: [10359] Moses, took his censer and stood between the living and...

  Overlap: 5 docs | Baseline-only: 0 | DEO-only: 0



### Example 2: "Provide an introduction to All Souls' Day, but do not mention Christianity"

Tests whether DEO can filter out Christianity-related content while keeping general information about All Souls' Day traditions.

In [ ]:
baseline_2, deo_2 = compare_results(
    query="Provide an introduction to All Souls' Day, but do not mention Christianity",
    positives=[
        "All Souls' Day traditions and customs",
        "Day of the Dead commemoration",
        "remembrance of deceased ancestors",
    ],
    negatives=[
        "Christianity",
        "Christian holiday",
        "Catholic Church liturgical calendar",
    ],
    k=5,
)

QUERY: Provide an introduction to All Souls' Day, but do not mention Christianity
  Positives: ["All Souls' Day traditions and customs", 'Day of the Dead commemoration', 'remembrance of deceased ancestors']
  Negatives: ['Christianity', 'Christian holiday', 'Catholic Church liturgical calendar']


INFO:deo_rag_with_bigquery.deo_optimizer:  DEO step   0 | loss=-0.0645 pos=0.7749 reg=0.0000 neg=0.8394
INFO:deo_rag_with_bigquery.deo_optimizer:  DEO step  19 | loss=-0.2406 pos=0.6211 reg=0.6471 neg=0.9911



           [Baseline Search]             |               [DEO Search]              
--------------------------------------------------------------------------------
  1. [10089] All Souls' Day In Christianity, All Souls' Day or the...
     → DEO: [10089] All Souls' Day In Christianity, All Souls' Day or the...

  2. [10088] the Church of England it is called The Commemoration of...
     → DEO: [10088] the Church of England it is called The Commemoration of...

  3. [11749] or Hell proper. Therefore, these souls neither merit the...
     → DEO: [13064] Afterlife The afterlife (also referred to as life after...

  4. [13064] Afterlife The afterlife (also referred to as life after...
     → DEO: [11500] move upward and leave the cycle of reincarnation. There...

  5. [12360] the dead go to a specific plane of existence after death,...
     → DEO: [13062] of the wicked is taken by the demon Vizaresa (Vīzarəša),...

  Overlap: 3 docs | Baseline-only: 2 | DEO-only: 2



### Example 3: "Introduce Shakespeare's tragedies, but do not mention Hamlet."

Tests exclusion of a specific well-known work while retaining other tragedies.

In [ ]:
baseline_3, deo_3 = compare_results(
    query="Introduce Shakespeare's tragedies, but do not mention Hamlet.",
    positives=[
        "Shakespeare tragedies",
        "Macbeth King Lear Othello",
        "Shakespearean tragic plays",
    ],
    negatives=[
        "Hamlet",
        "Prince of Denmark",
        "Hamlet Shakespeare play",
    ],
    k=5,
)

QUERY: Introduce Shakespeare's tragedies, but do not mention Hamlet.
  Positives: ['Shakespeare tragedies', 'Macbeth King Lear Othello', 'Shakespearean tragic plays']
  Negatives: ['Hamlet', 'Prince of Denmark', 'Hamlet Shakespeare play']


INFO:deo_rag_with_bigquery.deo_optimizer:  DEO step   0 | loss=-0.1081 pos=0.7094 reg=0.0000 neg=0.8174
INFO:deo_rag_with_bigquery.deo_optimizer:  DEO step  19 | loss=-0.2372 pos=0.5803 reg=0.5782 neg=0.9331



           [Baseline Search]             |               [DEO Search]              
--------------------------------------------------------------------------------
  1. [10400] Shakespeare's tragedies explore the complexity of human...
     → DEO: [10400] Shakespeare's tragedies explore the complexity of human...

  2. [10530] Shakespeare's plays, such as Macbeth, King Lear, and...
     → DEO: [10530] Shakespeare's plays, such as Macbeth, King Lear, and...

  3. [10532] Shakespeare's Macbeth explores the corrupting influence...
     → DEO: [10532] Shakespeare's Macbeth explores the corrupting influence...

  4. [10401] Hamlet is one of Shakespeare's most famous tragedies,...
     → DEO: [10505] Shakespeare’s impact on literature is immense,...

  5. [10533] Shakespeare’s Hamlet and Marlowe’s Doctor Faustus both...
     → DEO: [10533] Shakespeare’s Hamlet and Marlowe’s Doctor Faustus both...

  Overlap: 4 docs | Baseline-only: 1 | DEO-only: 1



### Example 4: Inspect full document content

Drill into a specific result to verify whether negated terms actually appear in the retrieved documents.

In [ ]:
import re

def check_negated_terms(docs, negated_terms: list[str], label: str = ""):
    """Check how many retrieved documents contain the negated terms."""
    print(f"--- {label} ---")
    for i, doc in enumerate(docs):
        content_lower = doc.page_content.lower()
        found = [t for t in negated_terms if t.lower() in content_lower]
        doc_id = doc.metadata.get("_id", "?")
        status = f"CONTAINS {found}" if found else "CLEAN"
        print(f"  Doc {i+1} [{doc_id}]: {status}")
    print()

# Example 1: Check for "Moses" in Aaron query results
negated = ["Moses"]
check_negated_terms(baseline_1, negated, label="Baseline - Aaron (negated: Moses)")
check_negated_terms(deo_1, negated, label="DEO      - Aaron (negated: Moses)")

# Example 3: Check for "Hamlet" in Shakespeare query results
negated = ["Hamlet"]
check_negated_terms(baseline_3, negated, label="Baseline - Shakespeare (negated: Hamlet)")
check_negated_terms(deo_3, negated, label="DEO      - Shakespeare (negated: Hamlet)")

--- Baseline - Aaron (negated: Moses) ---
  Doc 1 [10001]: CONTAINS ['Moses']
  Doc 2 [11182]: CONTAINS ['Moses']
  Doc 3 [10359]: CONTAINS ['Moses']
  Doc 4 [10000]: CLEAN
  Doc 5 [10313]: CLEAN

--- DEO      - Aaron (negated: Moses) ---
  Doc 1 [10001]: CONTAINS ['Moses']
  Doc 2 [10000]: CLEAN
  Doc 3 [10313]: CLEAN
  Doc 4 [11182]: CONTAINS ['Moses']
  Doc 5 [10359]: CONTAINS ['Moses']

--- Baseline - Shakespeare (negated: Hamlet) ---
  Doc 1 [10400]: CLEAN
  Doc 2 [10530]: CLEAN
  Doc 3 [10532]: CONTAINS ['Hamlet']
  Doc 4 [10401]: CONTAINS ['Hamlet']
  Doc 5 [10533]: CONTAINS ['Hamlet']

--- DEO      - Shakespeare (negated: Hamlet) ---
  Doc 1 [10400]: CLEAN
  Doc 2 [10530]: CLEAN
  Doc 3 [10532]: CONTAINS ['Hamlet']
  Doc 4 [10505]: CONTAINS ['Hamlet']
  Doc 5 [10533]: CONTAINS ['Hamlet']



## References

- **DEO Paper**: [DEO: Training-Free Direct Embedding Optimization for Negation-Aware Retrieval](https://arxiv.org/abs/2603.09185)
    - [(GitHub) DEO-negation-aware-retrieval](https://github.com/taegyeong-lee/DEO-negation-aware-retrieval)